# Des scènes au GeoTIFF multibande

**Cartomize · Version 1.0**

Durée indicative : 30 minutes.

## Objectifs

Déclarer les rôles des bandes, gérer le recouvrement, extraire une emprise et séparer les produits scientifiques et visuels.

## Apport de Cartomize

Le prétraitement regroupe les bandes par scène, aligne les grilles et conserve un manifeste des opérations et de la couverture.

Données : paysage pédagogique généré avec une graine fixe. Les cartes et mesures ne décrivent aucun site réel.

## Préparation

Installer l’environnement décrit dans le [guide de démarrage](../README.md), puis exécuter les cellules dans l’ordre. Chaque exécution utilise un nouveau dossier de résultats.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
from IPython.display import display, Image
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'course.py').is_file() or (p/'tutorials/course.py').is_file())
root = root if (root/'course.py').is_file() else root/'tutorials'
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import cartomize as cm
from course import demo, run_directory, show_raster, show_rgb, gallery, CRS, CLASSES, ROLES

In [2]:
out = run_directory("02_pretraitement_multispectral")
d = demo(out/"data")

## 1. Décrire les scènes

Pour Landsat C2 L2 et Sentinel-2 L2A, la reconnaissance exploite les noms et métadonnées. Pour un autre produit, utiliser un manifeste explicite. Ici, les valeurs sont déjà des réflectances simulées : facteur 1 et décalage 0.

In [3]:
scenes = cm.load_scenes_manifest(d['scenes'])
first = scenes[0]
example_band = cm.Band(first.bands['red'].path, scale=1, offset=0, unit='reflectance')
example_scene = cm.Scene('exemple', {'red': example_band, 'nir': first.bands['nir']}, sensor='synthetic', acquired='2026-01-01')
pd.DataFrame([{'scène': s.scene_id, 'bandes': len(s.bands), 'date': s.acquired} for s in scenes])

Out[3]: 
         scène  bandes        date
0  synthetic_A       7  2026-01-01
1  synthetic_B       7  2026-01-01


## 2. Paramétrer les sorties

`mosaic=True` assemble les scènes compatibles. `mosaic=False` les traite séparément. `separate_bands=True` ajoute un GeoTIFF par bande. L’ordre `nir, red, green` affecte ces bandes aux canaux rouge, vert et bleu de l’affichage.

In [4]:
produits = cm.process_imagery(
    scenes, out/'preparation', aoi=d['aoi'], mosaic=True,
    multiband=True, separate_bands=True, composition=('nir','red','green'),
    band_order=ROLES, target_crs=CRS, resolution=30,
    mask_clouds=False, overlap='first',
)
record = produits.products[0]
scientific = Path(record['multiband'])
display(pd.DataFrame([{'produit': k, 'valeur': str(v)} for k,v in record.items() if k!='bands']))

,produit,valeur
0,scenes,"['synthetic_A', 'synthetic_B']"
1,multiband,/workspace/scratch/993477e68500/cartomize-work...
2,composition,/workspace/scratch/993477e68500/cartomize-work...
3,source_index,/workspace/scratch/993477e68500/cartomize-work...
4,report,/workspace/scratch/993477e68500/cartomize-work...


In [5]:
rgba, extent, stretch = cm.read_rgb(scientific, bands=('nir','red','green'))
fig,ax=plt.subplots(figsize=(9,5),layout='constrained')
ax.imshow(rgba,extent=extent);ax.set_title('Composition infrarouge : paysage simulé');ax.set_axis_off()
gallery(fig,'02_composite');plt.show()

## 3. Contrôler la couverture et les bandes

Les trous sont conservés comme NoData. Une mosaïque ne restitue pas une observation absente. Un masque de nuages demande une bande QA/SCL compatible ; désactiver ce masque n’effectue aucune correction des nuages.

In [6]:
standalone = cm.prepare_imagery(scenes, out/'assemblage.tif', aoi=d['aoi'], band_order=ROLES, mask_clouds=False)
separees = cm.split_bands(scientific, out/'bandes_seules')
cm.color_composite(scientific, out/'couleurs_naturelles.tif', bands='natural')
with rasterio.open(scientific) as src:
    assert src.count == len(ROLES)
    assert src.crs.to_string() == CRS
    assert src.descriptions == ROLES
    assert len(separees) == src.count
    assert np.isclose(src.res[0],30)
display(standalone.report)

{'version': '1.0.1',
 'product': 'tutorials/outputs/02_pretraitement_multispectral_fe0nnnv1/assemblage.tif',
 'source_index': 'tutorials/outputs/02_pretraitement_multispectral_fe0nnnv1/assemblage_source_index.tif',
 'band_order': ('blue', 'green', 'red', 'nir', 'swir1', 'swir2', 'rededge1'),
 'crs': 'PROJCS["WGS 84 / UTM zone 33S",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",15],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",10000000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","32733"]]',
 'resolution_m': 30.0,
 'width': 118,
 'height': 87,
 'transform': [30.0, 0.0, 300150.0, 0.

## 4. Relier les options aux contrôles de la fenêtre

Dans **Prétraitement multispectral**, charger les bandes ou scènes, vérifier leurs correspondances, sélectionner la zone d’étude, puis cocher Mosaïque, Multibande, Bandes séparées et Composition colorée selon les sorties souhaitées. Les canaux RVB concernent l’affichage. Le multibande reste l’entrée des indices et de la classification.

## 5. Appliquer un masque de qualité explicite

Un masque synthétique marque une zone invalide. Le test vérifie que ces pixels restent absents après préparation. Les valeurs d’un masque QA/SCL réel suivent la documentation du capteur ; elles ne sont pas assimilables à ce masque binaire.

In [7]:
from course import write_raster
with rasterio.open(first.bands['red'].path) as src:
    valid=np.ones(src.shape,dtype='uint8');valid[30:40,30:40]=0
    quality=write_raster(out/'validite.tif',valid,transform=src.transform,crs=src.crs)
masked_scene=cm.Scene('scene_avec_masque',first.bands,sensor='synthetic',acquired=first.acquired,
                      quality=quality,quality_kind='valid_mask')
masked_image=cm.prepare_imagery([masked_scene],out/'scene_masquee.tif',mask_clouds=True)
with rasterio.open(masked_image.path) as src:
    assert np.ma.getmaskarray(src.read(1,masked=True))[30:40,30:40].all()
print('Les 100 pixels du masque de qualité sont exclus.')

Les 100 pixels du masque de qualité sont exclus.


## Exercice

Produire les deux scènes séparément. Le nombre de produits principaux est-il identique à celui de la mosaïque ?

<details>
<summary>Éléments de correction</summary>

Avec `mosaic=False`, `produits.products` comporte deux groupes, chacun avec son multibande et les sorties demandées. Avec `mosaic=True`, un seul groupe est produit.

</details>

## Reproduction

Les paramètres, la graine et les chemins sont explicites dans les cellules. Le répertoire `out` contient les produits de cette exécution. Adapter les sources, le système de coordonnées et les paramètres avant de transférer la méthode à une étude.